In [1]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# Use the first uploaded CSV file
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Clean column names
df.columns = df.columns.str.strip()

required_columns = [
    "ID",
    "section_header",
    "section_text",
    "dialogue"
]

missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Remove completely empty records and normalize text fields
df = df.dropna(how="all", subset=required_columns).copy()

for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print(f"Loaded {len(df)} source records.")
display(df.head())

Saving MTS-Dialog-TrainingSet.csv to MTS-Dialog-TrainingSet.csv
Loaded 1201 source records.


,ID,section_header,section_text,dialogue
0,0,GENHX,The patient is a 76-year-old white female who ...,Doctor: What brings you back into the clinic t...
1,1,GENHX,The patient is a 25-year-old right-handed Cauc...,Doctor: How're you feeling today? \r\nPatient...
2,2,GENHX,"This is a 22-year-old female, who presented to...","Doctor: Hello, miss. What is the reason for yo..."
3,3,MEDICATIONS,Prescribed medications were Salmeterol inhaler...,Doctor: Are you taking any over the counter me...
4,4,CC,"Burn, right arm.","Doctor: Hi, how are you? \r\nPatient: I burned..."


In [2]:
import re
import hashlib
import json
from pathlib import Path

# Output paths
OUTPUT_DIR = Path("/content/mts_dialog_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = OUTPUT_DIR / "dataset.csv"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

# Split configuration
DEV_RATIO = 0.20
RANDOM_SEED = 42

# Question source:
# "dialogue" uses the complete dialogue field as the query.
# Change this if you have a separate extracted question field.
QUESTION_SOURCE = "dialogue"

# Keep False until gold answers and scenario labels are verified.
STRICT_VALIDATION = False

ALLOWED_SCENARIOS = {
    "A_SUFFICIENT",
    "B_MISSING",
    "C_DISTRACTING"
}

ALLOWED_SPLITS = {"dev", "test"}

In [3]:
def make_question_id(source_id: str) -> str:
    """
    Create a deterministic ID from the original source ID.
    """
    source_id = str(source_id).strip()

    if not source_id:
        raise ValueError("Encountered a record with an empty source ID.")

    digest = hashlib.sha256(
        source_id.encode("utf-8")
    ).hexdigest()[:16].upper()

    return f"QA_MED_{digest}"


def assign_split(question_id: str) -> str:
    """
    Deterministically assign a record to dev or test.
    """
    digest = hashlib.sha256(
        f"{RANDOM_SEED}:{question_id}".encode("utf-8")
    ).hexdigest()

    fraction = int(digest[:8], 16) / 0xFFFFFFFF

    return "dev" if fraction < DEV_RATIO else "test"


def build_context(section_header: str, section_text: str) -> str:
    """
    Combine section metadata and narrative without adding
    empty or duplicated sections.
    """
    header = section_header.strip()
    narrative = section_text.strip()

    if header and narrative:
        return f"{header}\n\n{narrative}"

    return header or narrative


def clean_text(text: str) -> str:
    """
    Normalize line endings and excess whitespace while
    preserving meaningful paragraph breaks.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


records = []

for _, row in df.iterrows():
    question_id = make_question_id(row["ID"])

    question = clean_text(row[QUESTION_SOURCE])

    context = build_context(
        row["section_header"],
        row["section_text"]
    )
    context = clean_text(context)

    if not question:
        raise ValueError(
            f"Empty question for source ID {row['ID']!r}. "
            "Inspect the source or extract the question first."
        )

    records.append({
        "question_id": question_id,
        "gold_answer": "",
        "scenario": "",
        "split": assign_split(question_id),
        "question": question,
        "context": context
    })

processed = pd.DataFrame(records)

# Verify that IDs are unique
if processed["question_id"].duplicated().any():
    duplicates = processed.loc[
        processed["question_id"].duplicated(keep=False),
        "question_id"
    ].tolist()

    raise ValueError(f"Duplicate question IDs found: {duplicates}")

print("Records processed:", len(processed))
print("\nSplit distribution:")
print(processed["split"].value_counts())
display(processed.head())

Records processed: 1201

Split distribution:
split
test    944
dev     257
Name: count, dtype: int64


,question_id,gold_answer,scenario,split,question,context
0,QA_MED_5FECEB66FFC86F38,,,test,Doctor: What brings you back into the clinic t...,GENHX\n\nThe patient is a 76-year-old white fe...
1,QA_MED_6B86B273FF34FCE1,,,test,Doctor: How're you feeling today? \nPatient: T...,GENHX\n\nThe patient is a 25-year-old right-ha...
2,QA_MED_D4735E3A265E16EE,,,test,"Doctor: Hello, miss. What is the reason for yo...","GENHX\n\nThis is a 22-year-old female, who pre..."
3,QA_MED_4E07408562BEDB8B,,,test,Doctor: Are you taking any over the counter me...,MEDICATIONS\n\nPrescribed medications were Sal...
4,QA_MED_4B227777D4DD1FC6,,,dev,"Doctor: Hi, how are you? \nPatient: I burned m...","CC\n\nBurn, right arm."


In [6]:
# Evaluation matrix
dataset = processed[
    ["question_id", "gold_answer", "scenario", "split"]
].copy()

# Clinical corpus
corpus = processed[
    ["question_id", "question", "context"]
].rename(
    columns={"question_id": "id"}
)

# Validate split values
assert set(dataset["split"]).issubset(ALLOWED_SPLITS)

# Save dataset as CSV
dataset.to_csv(
    DATASET_PATH,
    index=False,
    encoding="utf-8"
)

# Save corpus as JSON Lines
with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    for record in corpus.to_dict(orient="records"):
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("Saved files:")
print(DATASET_PATH)
print(CORPUS_PATH)

print("\nDataset preview:")
display(dataset.head())

print("\nFirst corpus record:")
print(json.dumps(
    corpus.iloc[0].to_dict(),
    indent=2,
    ensure_ascii=False
))

Saved files:
/content/mts_dialog_processed/dataset.csv
/content/mts_dialog_processed/corpus.jsonl

Dataset preview:


,question_id,gold_answer,scenario,split
0,QA_MED_5FECEB66FFC86F38,,,test
1,QA_MED_6B86B273FF34FCE1,,,test
2,QA_MED_D4735E3A265E16EE,,,test
3,QA_MED_4E07408562BEDB8B,,,test
4,QA_MED_4B227777D4DD1FC6,,,dev



First corpus record:
{
  "id": "QA_MED_5FECEB66FFC86F38",
  "question": "Doctor: What brings you back into the clinic today, miss? \nPatient: I came in for a refill of my blood pressure medicine. \nDoctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones. Have you noticed any changes or do you have any concerns regarding these issues? \nPatient: No. \nDoctor: Have you had any fever or chills, cough, congestion, nausea, vomiting, chest pain, chest pressure?\nPatient: No. \nDoctor: Great. Also, for our records, how old are you and what race do you identify yourself as?\nPatient: I am seventy six years old and identify as a white female.",
  "context": "GENHX\n\nThe patient is a 76-year-old white female who presents to the clinic today originally for hypertension and a med check. She has a history of hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic r

In [7]:
# Check exact headers
expected_dataset_columns = [
    "question_id",
    "gold_answer",
    "scenario",
    "split"
]

expected_corpus_columns = [
    "id",
    "question",
    "context"
]

assert list(dataset.columns) == expected_dataset_columns
assert list(corpus.columns) == expected_corpus_columns

# Check ID correspondence
assert dataset["question_id"].tolist() == corpus["id"].tolist()

# Reload saved files to validate serialization
dataset_check = pd.read_csv(
    DATASET_PATH,
    dtype=str,
    keep_default_na=False
)

corpus_check = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON on line {line_number}"
            ) from exc

        if set(record.keys()) != {"id", "question", "context"}:
            raise ValueError(
                f"Unexpected fields on line {line_number}"
            )

        corpus_check.append(record)

assert len(dataset_check) == len(corpus_check)

assert (
    dataset_check["question_id"].tolist()
    == [r["id"] for r in corpus_check]
)

print("All structural checks passed.")
print("Rows in dataset.csv:", len(dataset_check))
print("Rows in corpus.jsonl:", len(corpus_check))

All structural checks passed.
Rows in dataset.csv: 1201
Rows in corpus.jsonl: 1201


In [8]:
from google.colab import files

files.download(str(DATASET_PATH))
files.download(str(CORPUS_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>